# TP 2 — CNN sur CIFAR-10 et augmentation de données

**Objectifs**

- Travailler sur des images couleur 32×32 plus complexes que MNIST.
- Construire un CNN à 3 blocs convolutifs.
- Comparer entraînement avec et sans augmentation de données.

**Durée indicative : 60 minutes.**

> Note : on utilise un sous-ensemble (5 000 train / 1 000 test) pour rester en quelques minutes sur CPU. L'objectif est pédagogique, pas la performance absolue. Le premier appel téléchargera CIFAR-10 (~170 Mo) dans `data/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "data"
CIFAR_CLASSES = [
    "avion",
    "voiture",
    "oiseau",
    "chat",
    "cerf",
    "chien",
    "grenouille",
    "cheval",
    "bateau",
    "camion",
]
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

## Exercice 1 — Charger CIFAR-10 sans augmentation

1. Crée un transform `Compose([ToTensor, Normalize(MEAN, STD)])`.
2. Charge CIFAR-10 train et test, prends 5 000 / 1 000 premiers exemples.
3. Crée les DataLoaders (batch 128 train, 256 test).
4. Affiche 10 exemples (en re-dénormalisant pour avoir une image lisible) avec leur classe.

<details>
<summary>💡 Coup de pouce — charger CIFAR-10 sans augmentation</summary>

**🎯 But :** charger un dataset image RGB et le normaliser pour préparer l'entraînement.

**Définir la normalisation**

```python
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
```

Ce sont les **stats réelles de CIFAR-10** (calculées sur tout le train set). Toujours utiliser les **vraies stats** du dataset utilisé, jamais (0.5, 0.5, 0.5) générique.

**Pipeline de transformations**

```python
from torchvision import transforms, datasets
transform = transforms.Compose([
    transforms.ToTensor(),                  # PIL → tenseur (C, H, W) ∈ [0, 1]
    transforms.Normalize(MEAN, STD),         # x = (x - mean) / std → centré-réduit
])

train = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
test  = datasets.CIFAR10('./data', train=False, download=True, transform=transform)
```

**Ré-afficher une image normalisée**

Pour ré-afficher correctement une image **après normalisation**, il faut **dénormaliser** :

```python
def show(img_tensor):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std  = torch.tensor(STD).view(3, 1, 1)
    img = img_tensor * std + mean             # dénormalise
    img = img.clamp(0, 1)                      # sécurité
    plt.imshow(img.permute(1, 2, 0))           # (C, H, W) → (H, W, C)
```

⚠️ **Trois points subtils** :
- `.view(3, 1, 1)` : broadcast les stats sur les axes spatiaux.
- `clamp(0, 1)` : protection contre les valeurs un peu hors plage après calcul flottant.
- `.permute(1, 2, 0)` : PyTorch utilise `(C, H, W)`, matplotlib veut `(H, W, C)`.

**DataLoader**

```python
from torch.utils.data import DataLoader
train_loader = DataLoader(train, batch_size=64, shuffle=True, num_workers=2)
test_loader  = DataLoader(test,  batch_size=256, shuffle=False, num_workers=2)
```

- `shuffle=True` sur train (important), `False` sur test.
- `batch_size` test plus grand (pas de gradient → pas de mémoire activée → on peut batcher plus).
- `num_workers=2` : chargement parallèle des batchs (accélère sur CPU rapide).

</details>

In [ ]:
# TODO


## Exercice 2 — Construire un CNN à 3 blocs

Architecture suggérée (inspirée VGG simplifié) :

- Conv(3→32, k=3, padding=1) → ReLU → Conv(32→32) → ReLU → Pool 2×2
- Conv(32→64) → ReLU → Conv(64→64) → ReLU → Pool 2×2
- Conv(64→128) → ReLU → Pool 2×2
- Flatten → Linear(128*4*4 → 128) → ReLU → Dropout(0.5) → Linear(128 → 10)

Vérifie que `model(torch.zeros(1, 3, 32, 32)).shape == (1, 10)`.

<details>
<summary>💡 Coup de pouce — CNN à 3 blocs pour CIFAR-10</summary>

**🎯 But :** construire un CNN plus profond qu'en TP1 MNIST, adapté à CIFAR-10 (32×32 RGB, 10 classes).

**Architecture cible : 3 blocs Conv + 1 FC**

```
Bloc 1: Conv(3→32) → BN → ReLU → Conv(32→32) → BN → ReLU → MaxPool(2)   # 32→16
Bloc 2: Conv(32→64) → BN → ReLU → Conv(64→64) → BN → ReLU → MaxPool(2)   # 16→8
Bloc 3: Conv(64→128) → BN → ReLU → Conv(128→128) → BN → ReLU → MaxPool(2) # 8→4
Flatten → Linear(128*4*4, 128) → ReLU → Dropout(0.5) → Linear(128, 10)
```

**Calcul des dimensions spatiales**

Entrée : 32×32 → MaxPool → 16×16 → MaxPool → 8×8 → MaxPool → 4×4.

⚠️ Tout le calcul du FC se base là-dessus :

```python
self.fc1 = nn.Linear(128 * 4 * 4, 128)   # ← 128 channels × 4 × 4 = 2048 features
```

Si vous changez le nombre de pools ou les paddings, ce calcul change.

**Pourquoi BatchNorm ?**

`nn.BatchNorm2d(channels)` normalise les activations dans le batch → entraînement **plus stable**, **plus rapide**, tolère des `learning_rate` plus grands. Standard dans tous les CNN modernes.

**Pourquoi Dropout ?**

```python
nn.Dropout(0.5)
```

Met aléatoirement 50 % des activations à zéro pendant l'entraînement → empêche le modèle de surdépendre de neurones individuels = régularisation. Se place typiquement **avant la dernière couche FC**.

⚠️ Dropout n'agit **qu'en mode `model.train()`**. En `model.eval()`, il est désactivé automatiquement.

**Compter les paramètres**

```python
n = sum(p.numel() for p in model.parameters())
print(f"{n:,} params")
```

Attendez-vous à **300k - 500k** paramètres. C'est petit pour CIFAR-10 mais suffisant pour démontrer le principe.

</details>

In [ ]:
# TODO


## Exercice 3 — Entraînement sans augmentation

Réutilise tes fonctions `train_one_epoch` et `evaluate` du TP 1 (ou recopie-les). Entraîne 3 époques avec `Adam(lr=1e-3)`. Conserve l'historique des accuracies.

In [ ]:
# TODO


## Exercice 4 — Avec augmentation de données

Crée un transform train avec augmentation :

```python
train_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
```

Recharge le dataset train avec ce transform (le test ne change pas). Ré-instancie un modèle frais, entraîne 3 époques. Compare les courbes accuracy avec et sans augmentation.

<details>
<summary>💡 Coup de pouce — augmentation de données sur CIFAR-10</summary>

**🎯 But :** ajouter de la diversité artificielle au train pour réduire l'overfitting.

**Transformations d'augmentation**

```python
train_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # crop décalé après padding 4 px
    transforms.RandomHorizontalFlip(),           # 50 % de chances de flip
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
```

**Pourquoi ces transformations ?**

- **`RandomCrop(32, padding=4)`** : ajoute 4 px de padding (zéro) puis recrop en 32×32 à une position aléatoire. Simule le **shift spatial** : une voiture peut être centrée, ou un peu à gauche, etc.
- **`RandomHorizontalFlip()`** : miroir horizontal aléatoire. OK pour CIFAR-10 (un avion vu de gauche ou droite reste un avion). ⚠️ Pas OK pour des chiffres / lettres (un « b » miroité devient « d »).

**Autres augmentations courantes**

- `ColorJitter(brightness=0.2, contrast=0.2)` : variations colorimétriques.
- `RandomRotation(15)` : ±15° (utile si l'angle réel varie).
- `RandomErasing()` : efface une zone aléatoire (simule occlusion).

⚠️ **Le test, lui, garde le transform original** (sans augmentation) :

```python
test_clean = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
```

Sinon vous évaluez sur des données déformées → métrique faussée.

**Recharger le dataset**

```python
train_aug_ds = datasets.CIFAR10('./data', train=True, transform=train_aug, download=False)
train_aug_loader = DataLoader(train_aug_ds, batch_size=64, shuffle=True, num_workers=2)
```

**Ré-entraîner et comparer**

Avec augmentation, vous devriez gagner **+2 à +5 points** d'accuracy test, surtout si vous entraînez plus longtemps (10-20 époques). L'augmentation a peu d'effet à 1-2 époques car le modèle n'a pas eu le temps de profiter de la diversité.

**Astuce — visualiser une image augmentée**

```python
img, _ = train_aug_ds[0]
show(img)   # voir l'effet du crop + flip
```

Tournez l'index plusieurs fois → vous voyez la même image avec des crops différents.

</details>

In [ ]:
# TODO
